# Task 3 — Review Summarization with Generative AI

**Goal:** one short blog-style article per product category (from Task 2): top 3 products and how they differ, top complaints, and the product to avoid.

**Approach: pandas finds the facts, a small LLM writes them up.**
1. Load `reviews_clustered.csv` (output of notebook 03).
2. Compute product stats per category with pandas: review count, average rating, % negative.
3. Pick the top 3, the weakest product and real complaint quotes, then build a short **facts sheet** per category.
4. Load a small instruct model (1.5B–3B parameters).
5. Write **3 prompt variants**, test them on one category, score them, pick the best.
6. Generate one article per category and save it to `outputs/articles/`.

Why not hand the LLM all the reviews? 28k reviews don't fit in a small model's context, and the model would invent facts.
A facts sheet keeps every number traceable to the data.

**Hardware:** a GPU makes this fast. Use Google Colab (Runtime → Change runtime type → T4 GPU) or run locally on a Mac (Apple GPU / CPU, slower).



In [5]:
import sys, re, json, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import sys
sys.path.append("..")


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: upload reviews_clustered.csv with the file panel on the left (or mount Drive).
    DATA_PATH = Path("reviews_clustered.csv")
    ARTICLES_DIR = Path("/content/drive/MyDrive/articles/my_model.csv")
else:
    sys.path.append("..")
    from src import config
    DATA_PATH = config.PROCESSED_DIR / "reviews_clustered.csv"
    ARTICLES_DIR = config.ARTICLES_DIR

ARTICLES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
print("Colab:", IN_COLAB, "| data:", DATA_PATH)

Colab: True | data: reviews_clustered.csv


In [6]:
import os
print(os.getcwd())
print(os.listdir())


/content
['.config', 'articles', 'drive', 'reviews_clustered.csv', 'sample_data']


In [7]:
path = "/content/drive/MyDrive/articles/test.txt"
with open(path, "w") as f: 
    f.write("Google Drive test successful!") 
print(f"File created at: {path}")

File created at: /content/drive/MyDrive/articles/test.txt


## Step 1. Load the clustered reviews
Each review already has `product` (clean name) and `category` (from Task 2).
If the categories still say "Cluster 0, 1, ...", name them in notebook 03 (Step 9) and re-run it first: the names go into the article titles.

In [6]:

reviews = pd.read_csv(DATA_PATH)
reviews = reviews.dropna(subset=["product", "category", "reviews.text", "reviews.rating"])
print(reviews.shape)

display(reviews["category"].value_counts())

if reviews["category"].str.match(r"^Cluster \d+$").all():
    print("Warning: categories are still placeholders. Name them in notebook 03, Step 9.")

(27863, 7)


,count
category,
Cluster 4,15630
Cluster 1,6555
Cluster 0,3795
Cluster 3,1590
Cluster 2,293


## Step 2. Product stats per category
One row per product. `pct_negative` = share of 1–2 star reviews.

Products with very few reviews are not ranked: an average of 5.0 from 3 reviews means nothing.
`MIN_REVIEWS` sets that bar; a category with too few products above it falls back to its most-reviewed products.

In [ ]:
MIN_REVIEWS = 30

stats = (
    reviews.groupby(["category", "product"])
    .agg(
        n_reviews=("reviews.text", "size"),
        avg_rating=("reviews.rating", "mean"),
        pct_negative=("reviews.rating", lambda r: (r <= 2).mean() * 100),
    )
    .reset_index()
    .round({"avg_rating": 2, "pct_negative": 1})
)

for cat, group in stats.groupby("category"):
    print(f"\n=== {cat}: {len(group)} products, {(group['n_reviews'] >= MIN_REVIEWS).sum()} with >= {MIN_REVIEWS} reviews")
    display(group.sort_values("n_reviews", ascending=False).head(8))

## Step 3. Build a facts sheet per category
For each category:
- **Ranked products**: products with enough reviews, sorted by average rating (ties: more reviews first).
- **Top 3**: the first three of that list.
- **Weakest**: the last one. Ratings in this dataset are all high (about 4.4–4.9), so "weakest" means relatively weakest, and the article should say so.
- **Complaints**: real quotes from 1–2 star reviews (short ones, 40–300 characters), topped up with 3-star quotes if a product has few bad reviews.
- **Praise**: a few 5-star quotes, so the model can describe what differs between the top 3.

In [ ]:
def ranked_products(cat, min_reviews=MIN_REVIEWS):
    """Products of one category with enough reviews, best first."""
    group = stats[stats["category"] == cat]
    ranked = group[group["n_reviews"] >= min_reviews]
    if len(ranked) < 3:                                   # small category: use the most-reviewed products instead
        ranked = group.nlargest(min(4, len(group)), "n_reviews")
    return ranked.sort_values(["avg_rating", "n_reviews"], ascending=[False, False])


def sample_quotes(product, rating_filter, n=4, min_len=40, max_len=300):
    """Short, distinct review quotes for one product."""
    texts = reviews.loc[(reviews["product"] == product) & rating_filter(reviews["reviews.rating"]), "reviews.text"]
    texts = texts.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    texts = texts[texts.str.len().between(min_len, max_len)].drop_duplicates()
    if texts.empty:
        return []
    return texts.sample(min(n, len(texts)), random_state=RANDOM_STATE).tolist()


def complaints(product, n=4):
    """1-2 star quotes first; top up with 3-star quotes (often mixed) only if needed."""
    quotes = sample_quotes(product, lambda r: r <= 2, n)
    if len(quotes) < n:
        quotes += sample_quotes(product, lambda r: r == 3, n - len(quotes))
    return quotes


def praise(product, n=2):
    return sample_quotes(product, lambda r: r == 5, n)


def product_line(row):
    return f"{row['product']} | {row['avg_rating']:.2f} stars | {row['n_reviews']} reviews | {row['pct_negative']:.1f}% negative"


def build_facts(cat):
    """Everything the LLM may use for one category, as plain text."""
    ranked = ranked_products(cat)
    top3 = ranked.head(3)
    weakest = ranked.iloc[-1] if len(ranked) > 3 else None

    lines = [f"CATEGORY: {cat}",
             f"Products in category: {(stats['category'] == cat).sum()}, total reviews: {int((reviews['category'] == cat).sum())}",
             "", "TOP 3 PRODUCTS (name | average rating | number of reviews | % negative reviews):"]
    for i, (_, row) in enumerate(top3.iterrows(), 1):
        lines.append(f"{i}. {product_line(row)}")
        for q in praise(row["product"]):
            lines.append(f"   + praise: \"{q}\"")
        for q in complaints(row["product"]):
            lines.append(f"   - complaint: \"{q}\"")

    lines.append("")
    if weakest is not None:
        lines.append("WEAKEST PRODUCT (lowest average rating among products with enough reviews):")
        lines.append(product_line(weakest))
        for q in complaints(weakest["product"], n=5):
            lines.append(f"   - complaint: \"{q}\"")
    else:
        lines.append("WEAKEST PRODUCT: not enough products with reviews to name one.")
    return "\n".join(lines)


categories = reviews["category"].value_counts().index.tolist()   # largest category first
facts = {cat: build_facts(cat) for cat in categories}
print(facts[categories[0]])

## Step 4. Load a small instruct model
Default: **`Qwen/Qwen2.5-1.5B-Instruct`** (no login needed, fast).
For better writing try `Qwen/Qwen2.5-3B-Instruct`, or `meta-llama/Llama-3.2-3B-Instruct` (needs a Hugging Face login and access request).

Why not T5 / BART? They shorten text well, but they can't follow a structure like *intro → top 3 → complaints → avoid*. An instruct model can.

If `transformers` is missing: `%pip install transformers accelerate`, then restart the kernel.

In [ ]:
import torch
from transformers import pipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    device, dtype = "cuda", torch.float16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.float32
print("Device:", device)

generator = pipeline("text-generation", model=MODEL_ID, torch_dtype=dtype, device=device)


def generate(messages, max_new_tokens=700):
    """Send chat messages to the model and return only its answer."""
    out = generator(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,             # same input -> same output, easier to compare prompts
        repetition_penalty=1.1,
    )
    return out[0]["generated_text"][-1]["content"].strip()

## Step 5. Three prompt variants
| Variant | What changes |
|---|---|
| **v1 basic** | One plain instruction |
| **v2 structured** | Fixed sections, word limit, "use only the facts" rule |
| **v3 structured + example** | v2 plus a short example article that shows the format (few-shot) |

In [ ]:
SYSTEM_RULES = (
    "You are a consumer-tech writer for a product review blog. "
    "Use ONLY the facts provided. Never invent products, numbers, features or prices. "
    "Quote ratings exactly as given."
)

SECTIONS = """Use exactly these Markdown sections:
# <catchy title that names the category>
## In short  (2 sentences)
## Top 3 picks  (one short paragraph per product: rating, number of reviews, what people like, how it differs from the others)
## Common complaints  (2-3 bullets per top product, based on the complaint quotes)
## The one to skip  (the weakest product and why; if ratings are all high, say it is the weakest of a strong group)
## Verdict  (1-2 sentences)
Length: 250-350 words."""

EXAMPLE_ARTICLE = """FORMAT EXAMPLE ONLY (made-up category, do not copy its products or numbers):
# Desk Lamps That Earn Their Spot
## In short
Buyers love the bright, adjustable lamps here. The cheaper models lose points for flimsy clamps.
## Top 3 picks
**LumaArc Pro (4.80 stars, 1,200 reviews)** - praised for even light ... Unlike the others, it ...
## Common complaints
- **LumaArc Pro:** the clamp loosens over time; ...
## The one to skip
**DimBright Mini (4.10 stars, 85 reviews)** - the weakest of a strong group: reviewers mention ...
## Verdict
Pick the LumaArc Pro unless ..."""

def prompt_v1(cat_facts):
    return [{"role": "user", "content": f"Write a short blog post about these products.\n\n{cat_facts}"}]

def prompt_v2(cat_facts):
    return [
        {"role": "system", "content": SYSTEM_RULES},
        {"role": "user", "content": f"Write a buyer's guide article from these facts.\n\n{SECTIONS}\n\nFACTS:\n{cat_facts}"},
    ]

def prompt_v3(cat_facts):
    return [
        {"role": "system", "content": SYSTEM_RULES},
        {"role": "user", "content": f"Write a buyer's guide article from these facts.\n\n{SECTIONS}\n\n{EXAMPLE_ARTICLE}\n\nFACTS:\n{cat_facts}"},
    ]

PROMPTS = {"v1_basic": prompt_v1, "v2_structured": prompt_v2, "v3_example": prompt_v3}

## Step 6. Test every variant on one category
Start small: one category, three variants. Read all three before scoring.

In [ ]:
TEST_CATEGORY = categories[0]    # largest category; change to test another one

drafts = {}
for name, make_prompt in PROMPTS.items():
    drafts[name] = generate(make_prompt(facts[TEST_CATEGORY]))
    display(Markdown(f"---\n### {name}\n\n{drafts[name]}"))

## Step 7. Score the variants
Automatic checks (fast, rough):
- **words**: is the length 250–350?
- **sections_found**: how many of the 5 required `##` sections appear?
- **top3_named**: how many of the top 3 products are mentioned (first 3 words of the name)?
- **invented_ratings**: ratings like `4.73` in the text that are not in the facts sheet (should be 0).

Then add your own 1–5 scores for readability and for whether the complaints match the quotes.

In [ ]:
REQUIRED_SECTIONS = ["In short", "Top 3 picks", "Common complaints", "The one to skip", "Verdict"]

def auto_checks(article, cat):
    ranked = ranked_products(cat)
    top3_names = [" ".join(p.split()[:3]).lower() for p in ranked.head(3)["product"]]
    allowed = {f"{r:.2f}" for r in stats.loc[stats["category"] == cat, "avg_rating"]}
    allowed |= {f"{r:.1f}" for r in stats.loc[stats["category"] == cat, "avg_rating"]}
    ratings_in_text = set(re.findall(r"\b[1-5]\.\d{1,2}\b", article))
    return {
        "words": len(article.split()),
        "sections_found": sum(s.lower() in article.lower() for s in REQUIRED_SECTIONS),
        "top3_named": sum(n in article.lower() for n in top3_names),
        "invented_ratings": len(ratings_in_text - allowed),
    }

scores = pd.DataFrame({name: auto_checks(text, TEST_CATEGORY) for name, text in drafts.items()}).T

# Your own judgement after reading the drafts (1 = bad, 5 = great)
scores["readability"] = pd.Series({"v1_basic": None, "v2_structured": None, "v3_example": None})
scores["complaints_match_quotes"] = pd.Series({"v1_basic": None, "v2_structured": None, "v3_example": None})
scores

## Step 8. Generate all articles with the best variant
Set `BEST_VARIANT` after reading Step 7, and write one sentence on why it won.
Each article is saved as Markdown, with the facts sheet below it, so anyone can check the numbers.

In [ ]:
BEST_VARIANT = "v2_structured"    # <-- change after Step 7

def slugify(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

articles = {}
for cat in categories:
    article = generate(PROMPTS[BEST_VARIANT](facts[cat]))
    articles[cat] = article
    path = ARTICLES_DIR / f"{slugify(cat)}.md"
    path.write_text(
        f"{article}\n\n---\n\n<details><summary>Facts sheet used ({MODEL_ID}, prompt {BEST_VARIANT})</summary>\n\n"
        f"```\n{facts[cat]}\n```\n</details>\n",
        encoding="utf-8",
    )
    print("Saved", path)

display(Markdown(articles[categories[0]]))

In [ ]:
# Quick check of every saved article
summary = pd.DataFrame({cat: auto_checks(text, cat) for cat, text in articles.items()}).T
summary

In [ ]:
# Colab only: download the articles as a zip, then copy them into outputs/articles/ in the repo
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/articles", "zip", ARTICLES_DIR)
    files.download("/content/articles.zip")

## Step 9 (optional). Fine-tune only if prompting is not enough
If the best prompt still invents facts or ignores the structure, try a bigger model first (3B), then consider LoRA fine-tuning (playbook section G).
For this project, prompt engineering is usually enough.

## Findings
Write 3–5 bullets after running:
- Model used and why:
- Best prompt variant and why it won (use the Step 7 table):
- Typical mistakes the model made (invented numbers? missing sections?):
- One thing the articles reveal about the categories: